## Precipitation from CHIRPS Daily (Google Earth Engine)

This notebook fetches **UCSB-CHG/CHIRPS/DAILY** precipitation and, for each sample in **water_quality_training_dataset.csv** (Latitude, Longitude, Sample Date), derives:

1. **Sum of precipitation in the 7 days before** the sampling date (days D-7 through D-1, mm).
2. **Number of days since last rain** (days since the most recent day with precipitation above a threshold, e.g. 0.1 mm).

### Prerequisites

1. **Install** the Earth Engine API: `pip install earthengine-api`
2. **Authenticate** (one-time): run the cell with `ee.Authenticate()`, then `ee.Initialize()`

### Dataset

[CHIRPS Daily](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY): daily precipitation (mm/day), 0.05° resolution (~5566 m), 1981–present, band `precipitation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta

import ee

### 1. Authenticate and initialize (run once)

Uncomment and run the authenticate cell once if needed, then run initialize.

In [ ]:
# One-time: opens browser to sign in with Google and grant Earth Engine access
# ee.Authenticate()

True

In [3]:
try:
    ee.Initialize()
    print("Earth Engine initialized.")
except Exception as e:
    print("Run ee.Authenticate() first, then ee.Initialize(). Error:", e)

Earth Engine initialized.


### 2. Load water quality training data (Latitude, Longitude, Sample Date)

We use **data/original/water_quality_training_dataset.csv**. Sample Date format is DD-MM-YYYY.

In [5]:
# Paths
if os.path.exists("data"):
    DATA_DIR = "data"
else:
    DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
ORIGINAL_DIR = os.path.join(DATA_DIR, "original")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

train_path = os.path.join(ORIGINAL_DIR, "water_quality_training_dataset.csv")
df = pd.read_csv(train_path)

# Parse Sample Date (DD-MM-YYYY)
df["sample_date"] = pd.to_datetime(df["Sample Date"], format="%d-%m-%Y")
print(f"Rows: {len(df)}, date range: {df['sample_date'].min()} to {df['sample_date'].max()}")

Rows: 9319, date range: 2011-01-02 00:00:00 to 2015-12-31 00:00:00


### 3. Parameters

CHIRPS scale ~5566 m. "7 days before" = D-7 through D-1 (inclusive). "Days since last rain" uses a rain threshold (mm) and a lookback window (e.g. 90 days).

In [6]:
CHIRPS_SCALE = 5566  # meters
DAYS_BEFORE_FOR_SUM = 7   # sum of precip in 7 days before sample date
LOOKBACK_DAYS = 90       # look back up to 90 days for "days since last rain"
RAIN_THRESHOLD_MM = 0.1  # day counts as "rain" if precip >= this (mm)

### 4. Process by unique sample date

For each unique **sample_date**:
1. **7-day sum**: Filter CHIRPS for (date - 7, date), sum to one image, sample at all points with that date.
2. **Days since last rain**: Filter CHIRPS for (date - lookback, date), get daily time series at all points via `getRegion`, then in Python compute days since last rain per point.

In [7]:
def get_precip_7d_sum_and_days_since_rain(df_train, chirps_scale=5566, days_before=7, lookback_days=90, rain_threshold_mm=0.1):
    """
    For each row in df_train (must have Latitude, Longitude, sample_date):
    - precip_7d_sum: sum of CHIRPS precipitation in the 7 days before sample_date (mm)
    - days_since_last_rain: number of days since most recent day with precip >= rain_threshold_mm (within lookback_days)
    """
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
    results = []
    unique_dates = df_train["sample_date"].drop_duplicates().sort_values()

    for sample_date in unique_dates:
        sample_date = pd.Timestamp(sample_date)
        date_str = sample_date.strftime("%Y-%m-%d")
        start_7d = (sample_date - timedelta(days=days_before)).strftime("%Y-%m-%d")
        start_lookback = (sample_date - timedelta(days=lookback_days)).strftime("%Y-%m-%d")
        end_date = sample_date.strftime("%Y-%m-%d")  # GEE end is exclusive, so we use sample_date as end to get up to sample_date-1

        # Points that have this sample_date
        mask = df_train["sample_date"] == sample_date
        sub = df_train.loc[mask].reset_index(drop=True)
        if sub.empty:
            continue

        lats = sub["Latitude"].tolist()
        lons = sub["Longitude"].tolist()
        coords = [[lon, lat] for lon, lat in zip(lons, lats)]

        # 1) 7-day sum image, sample at points
        chirps_7d = chirps.filterDate(start_7d, end_date).sum()
        points_fc = ee.FeatureCollection([
            ee.Feature(ee.Geometry.Point(c), {"id": j})
            for j, c in enumerate(coords)
        ])
        reduced_7d = chirps_7d.reduceRegions(
            collection=points_fc,
            reducer=ee.Reducer.first(),
            scale=chirps_scale,
        )
        info_7d = reduced_7d.getInfo()
        precip_7d_by_id = {}
        for f in info_7d.get("features", []):
            props = f.get("properties", {})
            idx = props.get("id")
            p = props.get("precipitation")
            precip_7d_by_id[idx] = float(p) if p is not None else np.nan

        # 2) Daily time series via getRegion (one point per geometry for MultiPoint returns one row per (point, image))
        chirps_lookback = chirps.filterDate(start_lookback, end_date)
        if coords:
            multi_point = ee.Geometry.MultiPoint(coords)
            try:
                region_data = chirps_lookback.getRegion(multi_point, chirps_scale).getInfo()
            except Exception as e:
                print(f"getRegion failed for {date_str}: {e}")
                region_data = None
        else:
            region_data = None

        # Parse getRegion: first row = headers, then [lon, lat, time, precipitation]
        days_since_by_coord = {}
        if region_data and len(region_data) > 1:
            headers = region_data[0]
            time_idx = headers.index("time") if "time" in headers else 2
            lon_idx = headers.index("longitude") if "longitude" in headers else 0
            lat_idx = headers.index("latitude") if "latitude" in headers else 1
            precip_idx = headers.index("precipitation") if "precipitation" in headers else 3
            # Group by (lon, lat) and collect (time, precip)
            by_point = {}
            for row in region_data[1:]:
                lon, lat = row[lon_idx], row[lat_idx]
                ts = row[time_idx]
                precip = row[precip_idx] if precip_idx < len(row) else None
                try:
                    precip = float(precip) if precip is not None else np.nan
                except (TypeError, ValueError):
                    precip = np.nan
                key = (round(float(lon), 6), round(float(lat), 6))
                if key not in by_point:
                    by_point[key] = []
                by_point[key].append((ts, precip))
            # Sample date - 1 (most recent day before sample)
            sample_day_end = sample_date - timedelta(days=1)
            for key, series in by_point.items():
                series.sort(key=lambda x: x[0], reverse=True)  # newest first
                days_since = np.nan
                for d, (ts, precip) in enumerate(series):
                    if precip is not np.nan and precip >= rain_threshold_mm:
                        days_since = d
                        break
                if np.isnan(days_since) and series:
                    days_since = lookback_days  # no rain in lookback
                days_since_by_coord[key] = days_since
        else:
            for i, (lon, lat) in enumerate(zip(lons, lats)):
                days_since_by_coord[(round(lon, 6), round(lat, 6))] = np.nan

        # Attach to each row for this date
        for j in range(len(sub)):
            row = sub.iloc[j]
            lon, lat = float(row["Longitude"]), float(row["Latitude"])
            key = (round(lon, 6), round(lat, 6))
            precip_7d = precip_7d_by_id.get(j, np.nan)
            days_since = days_since_by_coord.get(key, np.nan)
            results.append({
                "Latitude": lat,
                "Longitude": lon,
                "Sample Date": row["Sample Date"],
                "sample_date": sample_date,
                "precip_7d_sum_mm": precip_7d,
                "days_since_last_rain": days_since,
            })
    return pd.DataFrame(results)

The loop above uses row index in `sub` for matching; we need to match by position (order in `sub`) for `precip_7d_by_id`. Fixing the indexing: `id` in the FeatureCollection is `i` from `enumerate(coords)`, so the k-th row in `sub` has `id = k`. So we use `precip_7d_by_id.get(i, np.nan)` where `i` is the row index in the loop. Actually we're iterating `for i, row in sub.iterrows()` so `i` is the original dataframe index, not 0,1,2. So we should enumerate(sub) to get 0,1,2 for id. Let me fix the code to use explicit index 0..len(sub)-1 for id and then when building results we iterate with enumerate(sub) and use j as id.

In [8]:
def get_precip_7d_sum_and_days_since_rain(df_train, chirps_scale=5566, days_before=7, lookback_days=90, rain_threshold_mm=0.1):
    """
    For each row in df_train (must have Latitude, Longitude, sample_date):
    - precip_7d_sum_mm: sum of CHIRPS precipitation in the 7 days before sample_date (mm)
    - days_since_last_rain: number of days since most recent day with precip >= rain_threshold_mm
    """
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
    results = []
    unique_dates = df_train["sample_date"].drop_duplicates().sort_values()

    for sample_date in unique_dates:
        sample_date = pd.Timestamp(sample_date)
        date_str = sample_date.strftime("%Y-%m-%d")
        start_7d = (sample_date - timedelta(days=days_before)).strftime("%Y-%m-%d")
        start_lookback = (sample_date - timedelta(days=lookback_days)).strftime("%Y-%m-%d")
        end_date = sample_date.strftime("%Y-%m-%d")

        mask = df_train["sample_date"] == sample_date
        sub = df_train.loc[mask].reset_index(drop=True)
        if sub.empty:
            continue

        lats = sub["Latitude"].tolist()
        lons = sub["Longitude"].tolist()
        coords = [[lon, lat] for lon, lat in zip(lons, lats)]

        # 1) 7-day sum
        chirps_7d = chirps.filterDate(start_7d, end_date).sum()
        points_fc = ee.FeatureCollection([
            ee.Feature(ee.Geometry.Point(c), {"id": j})
            for j, c in enumerate(coords)
        ])
        reduced_7d = chirps_7d.reduceRegions(
            collection=points_fc,
            reducer=ee.Reducer.first(),
            scale=chirps_scale,
        )
        info_7d = reduced_7d.getInfo()
        precip_7d_by_id = {}
        for f in info_7d.get("features", []):
            props = f.get("properties", {})
            idx = props.get("id")
            p = props.get("precipitation")
            precip_7d_by_id[idx] = float(p) if p is not None else np.nan

        # 2) Daily time series for days since last rain
        chirps_lookback = chirps.filterDate(start_lookback, end_date)
        days_since_by_id = {j: np.nan for j in range(len(coords))}
        if coords:
            multi_point = ee.Geometry.MultiPoint(coords)
            try:
                region_data = chirps_lookback.getRegion(multi_point, chirps_scale).getInfo()
            except Exception as e:
                print(f"getRegion failed for {date_str}: {e}")
                region_data = None
            else:
                if region_data and len(region_data) > 1:
                    headers = region_data[0]
                    time_idx = headers.index("time") if "time" in headers else 2
                    lon_idx = headers.index("longitude") if "longitude" in headers else 0
                    lat_idx = headers.index("latitude") if "latitude" in headers else 1
                    precip_idx = next((h for h in range(len(headers)) if "precipitation" in str(headers[h]).lower()), 3)
                    if precip_idx >= len(headers):
                        precip_idx = 3
                    by_point = {}
                    for row in region_data[1:]:
                        lon, lat = float(row[lon_idx]), float(row[lat_idx])
                        ts = row[time_idx]
                        precip = row[precip_idx] if precip_idx < len(row) else None
                        try:
                            precip = float(precip) if precip is not None else np.nan
                        except (TypeError, ValueError):
                            precip = np.nan
                        key = (round(lon, 6), round(lat, 6))
                        if key not in by_point:
                            by_point[key] = []
                        by_point[key].append((ts, precip))
                    for j, (lon, lat) in enumerate(zip(lons, lats)):
                        key = (round(lon, 6), round(lat, 6))
                        series = by_point.get(key, [])
                        series.sort(key=lambda x: x[0], reverse=True)
                        for d, (ts, precip) in enumerate(series):
                            if precip is not np.nan and precip >= rain_threshold_mm:
                                days_since_by_id[j] = d
                                break
                        if np.isnan(days_since_by_id[j]) and series:
                            days_since_by_id[j] = lookback_days

        for j, (_, row) in enumerate(sub.iterrows()):
            results.append({
                "Latitude": row["Latitude"],
                "Longitude": row["Longitude"],
                "Sample Date": row["Sample Date"],
                "sample_date": sample_date,
                "precip_7d_sum_mm": precip_7d_by_id.get(j, np.nan),
                "days_since_last_rain": days_since_by_id.get(j, np.nan),
            })
    return pd.DataFrame(results)

In [9]:
# Run the extraction (this may take a while: one reduceRegions + one getRegion per unique sample date)
from tqdm import tqdm

unique_dates = df["sample_date"].drop_duplicates()
print(f"Unique sample dates: {len(unique_dates)}")

precip_df = get_precip_7d_sum_and_days_since_rain(
    df,
    chirps_scale=CHIRPS_SCALE,
    days_before=DAYS_BEFORE_FOR_SUM,
    lookback_days=LOOKBACK_DAYS,
    rain_threshold_mm=RAIN_THRESHOLD_MM,
)
print(f"Result rows: {len(precip_df)}")
precip_df.head(10)

Unique sample dates: 1364
Result rows: 9319


,Latitude,Longitude,Sample Date,sample_date,precip_7d_sum_mm,days_since_last_rain
0,-28.760833,17.730278,02-01-2011,2011-01-02,NaN,NaN
1,-26.861111,28.884722,03-01-2011,2011-01-03,NaN,NaN
2,-26.450000,28.085833,03-01-2011,2011-01-03,NaN,NaN
3,-27.671111,27.236944,03-01-2011,2011-01-03,NaN,NaN
4,-27.356667,27.286389,03-01-2011,2011-01-03,NaN,NaN
5,-27.010111,26.698083,04-01-2011,2011-01-04,NaN,NaN
6,-25.127778,27.628889,04-01-2011,2011-01-04,NaN,NaN
7,-25.206390,27.558000,04-01-2011,2011-01-04,NaN,NaN
8,-24.695140,27.409060,04-01-2011,2011-01-04,NaN,NaN
9,-26.984722,26.632278,04-01-2011,2011-01-04,NaN,NaN


In [ ]:
# Merge back to training dataframe on (Latitude, Longitude, Sample Date)
df_merged = df.merge(
    precip_df[["Latitude", "Longitude", "Sample Date", "precip_7d_sum_mm", "days_since_last_rain"]],
    on=["Latitude", "Longitude", "Sample Date"],
    how="left",
)
df_merged.head(10)

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
out_path = os.path.join(PROCESSED_DIR, "precipitation_chirps_training.csv")
precip_df.to_csv(out_path, index=False)
print(f"Saved {len(precip_df)} rows to {out_path}")

### Notes

- **7 days before**: CHIRPS images for (sample_date - 7) through (sample_date - 1) are summed; sample date itself is excluded.
- **Days since last rain**: We look back up to `LOOKBACK_DAYS` (90); a day counts as "rain" if precipitation ≥ `RAIN_THRESHOLD_MM` (0.1 mm). If no rain in the lookback, we set days_since_last_rain = lookback_days.
- **Performance**: One `reduceRegions` (getInfo) and one `getRegion` (getInfo) per unique sample date. For many unique dates, consider batching or exporting from GEE.
- **getRegion**: For MultiPoint, GEE returns one row per (point, image). Column order may vary; we detect "longitude", "latitude", "time", "precipitation" from the header row.